# Toolset Reliability Sensitivity Ablation (HF + Kaggle)

This notebook is a clean, standalone Toolset Reliability study runner focused on tool-calling sensitivity.

It intentionally leaves existing notebooks untouched and reuses repository scripts/helpers.

## Study intent
- Validate whether a finetuned adapter changes tool-calling behavior on target toolsets
- Run a sensitivity-oriented ablation config (stronger schema/timeout pressure + model-sensitive policy subset)
- Publish reproducible artifacts to Kaggle dataset

In [1]:
!nvidia-smi

Fri Apr  3 07:52:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import time
from pathlib import Path
from getpass import getpass

REPO_NAME = 'tool-calling-reliability-benchmark'
REPO_URL = 'https://github.com/aaliyan1230/tool-calling-reliability-benchmark.git'

def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    return None

repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    kaggle_repo = Path('/kaggle/working') / REPO_NAME
    if not kaggle_repo.exists():
        print(f'[setup] Cloning repo to {kaggle_repo} ...')
        subprocess.run(['git', 'clone', REPO_URL, str(kaggle_repo)], check=True)
    repo_root = kaggle_repo

REPO_ROOT = repo_root.resolve()
os.chdir(REPO_ROOT)

if shutil.which('uv') is None:
    print('[setup] Installing uv ...')
    subprocess.run(['python', '-m', 'pip', 'install', '-q', 'uv'], check=True)

print('Repo root:', REPO_ROOT)
print('Kernel cwd:', Path.cwd())

[setup] Cloning repo to /kaggle/working/tool-calling-reliability-benchmark ...


Cloning into '/kaggle/working/tool-calling-reliability-benchmark'...


Repo root: /kaggle/working/tool-calling-reliability-benchmark
Kernel cwd: /kaggle/working/tool-calling-reliability-benchmark


In [7]:
HF_TOKEN = str(os.environ.get('HF_TOKEN', '')).strip()
if not HF_TOKEN:
    HF_TOKEN = getpass('Enter HF_TOKEN (input hidden): ').strip()
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is required.')
os.environ['HF_TOKEN'] = HF_TOKEN

KAGGLE_USERNAME = str(os.environ.get('KAGGLE_USERNAME', '')).strip()
if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = input('Enter KAGGLE_USERNAME: ').strip()

KAGGLE_KEY = str(os.environ.get('KAGGLE_KEY', '')).strip()
if not KAGGLE_KEY:
    KAGGLE_KEY = str(os.environ.get('KAGGLE_API_TOKEN', '')).strip()
if not KAGGLE_KEY:
    KAGGLE_KEY = getpass('Enter KAGGLE_KEY (input hidden): ').strip()

if not KAGGLE_USERNAME or not KAGGLE_KEY:
    raise RuntimeError('KAGGLE_USERNAME and KAGGLE_KEY are required.')

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

try:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('[auth] Hugging Face login succeeded.')
except Exception as exc:
    print('[auth] HF login warning:', exc)

print('[auth] Kaggle credentials configured for runtime.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[auth] Hugging Face login succeeded.
[auth] Kaggle credentials configured for runtime.


In [8]:
LABEL_PREFIX = 'toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1'
SEEDS = '11,22,33,44,55,66,77,88,99,111'
BASE_PLANNER = 'configs/planners/hf_qwen2_5_3b_base.json'
FT_PLANNER = 'configs/planners/hf_qwen2_5_3b_ft.json'
BASE_CONFIG = REPO_ROOT / 'configs' / 'baseline.json'
SENSITIVITY_CONFIG = REPO_ROOT / 'configs' / 'toolset_reliability_sensitivity_v1.json'

ADAPTER_DATASET = 'aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts'
PUBLISH_DATASET_SLUG = 'tcrb-qwen25-3b-toolset-reliability-sensitivity-v1'
PUBLISH_DATASET_TITLE = 'TCRB Qwen2.5-3B Toolset Reliability Sensitivity V1'

print('LABEL_PREFIX =', LABEL_PREFIX)
print('SEEDS =', SEEDS)
print('BASE_CONFIG =', BASE_CONFIG)
print('SENSITIVITY_CONFIG =', SENSITIVITY_CONFIG)

LABEL_PREFIX = toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1
SEEDS = 11,22,33,44,55,66,77,88,99,111
BASE_CONFIG = /kaggle/working/tool-calling-reliability-benchmark/configs/baseline.json
SENSITIVITY_CONFIG = /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_sensitivity_v1.json


In [9]:
cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))
cfg['policies'] = ['naive_retry', 'exponential_backoff_jitter']
faults = dict(cfg.get('fault_probabilities', {}))
faults['malformed_schema'] = max(float(faults.get('malformed_schema', 0.06)), 0.12)
faults['timeout'] = max(float(faults.get('timeout', 0.08)), 0.12)
cfg['fault_probabilities'] = faults
cfg['max_attempts'] = max(int(cfg.get('max_attempts', 4)), 5)
cfg['time_budget_ms'] = max(int(cfg.get('time_budget_ms', 1800)), 2200)
SENSITIVITY_CONFIG.parent.mkdir(parents=True, exist_ok=True)
SENSITIVITY_CONFIG.write_text(json.dumps(cfg, indent=2) + '\n', encoding='utf-8')
print('Wrote sensitivity config:', SENSITIVITY_CONFIG)

Wrote sensitivity config: /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_sensitivity_v1.json


In [10]:
pull_script = REPO_ROOT / 'scripts' / 'pull_kaggle_adapter.py'
pull_cmd = ['uv', 'run', 'python', str(pull_script), '--dataset', ADAPTER_DATASET, '--repo-root', '.']
print('Running:', ' '.join(pull_cmd))
res = subprocess.run(pull_cmd, text=True, capture_output=True, check=False)
if res.stdout:
    print(res.stdout)
if res.returncode != 0:
    if res.stderr:
        print(res.stderr)
    raise RuntimeError(f'Adapter pull failed with code {res.returncode}')

Running: uv run python /kaggle/working/tool-calling-reliability-benchmark/scripts/pull_kaggle_adapter.py --dataset aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts --repo-root .
Dataset URL: https://www.kaggle.com/datasets/aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts
License(s): CC0-1.0

{
  "dataset": "aaliyanshaikh/tcrb-qwen25-3b-adapter-artifacts",
  "download_dir": "/kaggle/working/tool-calling-reliability-benchmark/tmp/kaggle_adapter_pull",
  "adapter_root": "/kaggle/working/tool-calling-reliability-benchmark/tmp/kaggle_adapter_pull/adapter",
  "target_dir": "/kaggle/working/tool-calling-reliability-benchmark/outputs/ft-notebook/final"
}
Adapter materialized successfully.



In [11]:
required_mods = ['torch', 'transformers', 'peft', 'trl', 'datasets', 'accelerate', 'bitsandbytes', 'wrapt']

probe_cmd = [

    'uv', 'run', 'python', '-c',

    "import importlib.util as u; mods=%r; missing=[m for m in mods if u.find_spec(m) is None]; print('MISSING=' + ','.join(missing))" % required_mods,

]

probe = subprocess.run(probe_cmd, text=True, capture_output=True, check=False)

probe_out = (probe.stdout or '').strip()

print('[deps] Probe output:', probe_out)



missing = []

if 'MISSING=' in probe_out:

    missing_text = probe_out.split('MISSING=', 1)[1].strip()

    if missing_text:

        missing = [m for m in missing_text.split(',') if m]



if missing:

    install_cmd = [

        'uv', 'pip', 'install', '--python', '.venv/bin/python',

        'torch', 'transformers', 'peft', 'trl', 'datasets', 'accelerate', 'bitsandbytes', 'wrapt',

    ]

    print('Running:', ' '.join(install_cmd))

    install = subprocess.run(install_cmd, text=True, capture_output=True, check=False)

    if install.stdout:

        print(install.stdout[-4000:])

    if install.returncode != 0:

        if install.stderr:

            print(install.stderr[-4000:])

        raise RuntimeError(f'uv pip install failed with code {install.returncode}')

    print('[deps] uv environment dependencies are ready.')

else:

    print('[deps] uv environment already has required modules.')

[deps] Probe output: MISSING=torch,transformers,peft,trl,datasets,accelerate,bitsandbytes,wrapt
Running: uv pip install --python .venv/bin/python torch transformers peft trl datasets accelerate bitsandbytes wrapt
[deps] uv environment dependencies are ready.


In [12]:
cmd = [
    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',
    '--config', str(SENSITIVITY_CONFIG),
    '--seeds', SEEDS,
    '--base-planner-config', BASE_PLANNER,
    '--ft-planner-config', FT_PLANNER,
    '--label-prefix', LABEL_PREFIX,
]
print('Running:', ' '.join(cmd))
started = time.time()
proc = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
print(f'[toolset-reliability] Total elapsed: {time.time() - started:.1f}s')
if rc != 0:
    raise RuntimeError(f'Study run failed with code {rc}')

Running: uv run python scripts/run_northstar_hf.py --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_sensitivity_v1.json --seeds 11,22,33,44,55,66,77,88,99,111 --base-planner-config configs/planners/hf_qwen2_5_3b_base.json --ft-planner-config configs/planners/hf_qwen2_5_3b_ft.json --label-prefix toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1
[northstar] HF_TOKEN set: True
[northstar] Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_sensitivity_v1.json --workload workloads/sample_tasks.json --seeds 11,22,33,44,55,66,77,88,99,111 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-base-ms

Fetching 2 files: 100%|██████████| 2/2 [00:22<00:00, 11.33s/it]

Loading weights: 100%|██████████| 434/434 [00:10<00:00, 40.81it/s]
Planner: hf_qwen2_5_3b_base
Wrote multi-seed results: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-

In [13]:
run_root = REPO_ROOT / 'runs'
base_ms = run_root / f'{LABEL_PREFIX}-base-ms' / 'multi_seed.json'
ft_ms = run_root / f'{LABEL_PREFIX}-ft-ms' / 'multi_seed.json'
matrix_json = run_root / f'{LABEL_PREFIX}-matrix' / 'matrix.json'
for p in [base_ms, ft_ms, matrix_json]:
    print('-', p, 'exists=' + str(p.exists()))
b = json.loads(base_ms.read_text(encoding='utf-8'))
f = json.loads(ft_ms.read_text(encoding='utf-8'))
m = json.loads(matrix_json.read_text(encoding='utf-8'))
by_b = {r['policy']: r['metrics'] for r in b.get('aggregate_policy_metrics', [])}
by_f = {r['policy']: r['metrics'] for r in f.get('aggregate_policy_metrics', [])}
policies = sorted(set(by_b) & set(by_f))
succ = [by_f[p]['task_success_rate']['mean'] - by_b[p]['task_success_rate']['mean'] for p in policies]
inv = [by_f[p]['invalid_tool_call_rate']['mean'] - by_b[p]['invalid_tool_call_rate']['mean'] for p in policies]
mean_success_delta = sum(succ) / len(succ) if succ else 0.0
mean_invalid_delta = sum(inv) / len(inv) if inv else 0.0
print('=== Toolset Reliability Summary ===')
print('policies:', policies)
print(f'mean success delta (ft-base): {mean_success_delta:+.4f}')
print(f'mean invalid delta (ft-base): {mean_invalid_delta:+.4f}')
print('matrix portfolio verdict:', m.get('portfolio_verdict'))

- /kaggle/working/tool-calling-reliability-benchmark/runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-base-ms/multi_seed.json exists=True
- /kaggle/working/tool-calling-reliability-benchmark/runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-ft-ms/multi_seed.json exists=True
- /kaggle/working/tool-calling-reliability-benchmark/runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-matrix/matrix.json exists=True
=== Toolset Reliability Summary ===
policies: ['exponential_backoff_jitter', 'naive_retry']
mean success delta (ft-base): -0.1167
mean invalid delta (ft-base): +0.0682
matrix portfolio verdict: FAIL


In [14]:
from pathlib import Path

import json
import subprocess


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate

    fallbacks = [
        Path('/kaggle/working/tool-calling-reliability-benchmark'),
        Path('/Users/aaliyan/aaliyan/tool-calling-reliability-benchmark'),
    ]
    for candidate in fallbacks:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    return None


repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    raise RuntimeError('Could not locate repository root for study-gate run.')
REPO_ROOT = repo_root.resolve()

try:
    LABEL_PREFIX
except NameError:
    LABEL_PREFIX = 'toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1'

run_root = REPO_ROOT / 'runs'
base_ms = run_root / f'{LABEL_PREFIX}-base-ms' / 'multi_seed.json'
ft_ms = run_root / f'{LABEL_PREFIX}-ft-ms' / 'multi_seed.json'
matrix_json = run_root / f'{LABEL_PREFIX}-matrix' / 'matrix.json'
gate_dir = run_root / f'{LABEL_PREFIX}-study-gate'
gate_dir.mkdir(parents=True, exist_ok=True)
gate_json = gate_dir / 'study_gate.json'
gate_md = gate_dir / 'study_gate.md'

cmd = [
    'uv', 'run', 'python', '-m', 'tcrb', 'study-gate',
    '--base-run', str(base_ms),
    '--finetuned-run', str(ft_ms),
    '--matrix-json', str(matrix_json),
    '--require-matrix-signal',
    '--require-matrix-not-fail',
    '--output-json', str(gate_json),
    '--output-report', str(gate_md),
]

print('Repo root:', REPO_ROOT)
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, text=True, cwd=str(REPO_ROOT), capture_output=True, check=False)
if res.stdout:
    print(res.stdout)
if res.returncode != 0 and res.stderr:
    print(res.stderr)
print('study-gate exit code =', res.returncode)

if gate_json.exists():
    payload = json.loads(gate_json.read_text(encoding='utf-8'))
    print('study-gate verdict =', payload.get('verdict'))
    for check in payload.get('checks', []):
        print('-', check.get('name'), 'PASS' if check.get('passed') else 'FAIL', 'value=', check.get('value'), 'threshold=', check.get('threshold'))
    print('study-gate markdown =', gate_md)
else:
    print('study-gate JSON was not generated:', gate_json)

Repo root: /kaggle/working/tool-calling-reliability-benchmark
Running: uv run python -m tcrb study-gate --base-run /kaggle/working/tool-calling-reliability-benchmark/runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-base-ms/multi_seed.json --finetuned-run /kaggle/working/tool-calling-reliability-benchmark/runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-ft-ms/multi_seed.json --matrix-json /kaggle/working/tool-calling-reliability-benchmark/runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-matrix/matrix.json --require-matrix-signal --require-matrix-not-fail --output-json /kaggle/working/tool-calling-reliability-benchmark/runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-study-gate/study_gate.json --output-report /kaggle/working/tool-calling-reliability-benchmark/runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-study-gate/study_gate.md
Wrote study gate JSON: /kaggle/working/tool-calling-reliability-benchmark/runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-study-gate/study_gate.j

## Calibrated Recovery Arm



This arm reduces stress relative to the sensitivity run and widens policy coverage to seek a practical, favorable outcome in the same runtime.


In [15]:
RECOVERY_LABEL_PREFIX = f"{LABEL_PREFIX}-recovery"

RECOVERY_SEEDS = '11,22,33,44,55'

RECOVERY_CONFIG = REPO_ROOT / 'configs' / 'toolset_reliability_recovery_v1.json'



recovery_cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))

recovery_cfg['policies'] = [

    'naive_retry',

    'exponential_backoff_jitter',

    'majority_vote',

    'self_consistency',

]

recovery_faults = dict(recovery_cfg.get('fault_probabilities', {}))

recovery_faults['malformed_schema'] = min(float(recovery_faults.get('malformed_schema', 0.06)), 0.06)

recovery_faults['timeout'] = min(float(recovery_faults.get('timeout', 0.08)), 0.08)

recovery_cfg['fault_probabilities'] = recovery_faults

recovery_cfg['max_attempts'] = max(int(recovery_cfg.get('max_attempts', 4)), 5)

recovery_cfg['time_budget_ms'] = max(int(recovery_cfg.get('time_budget_ms', 1800)), 2200)



RECOVERY_CONFIG.write_text(json.dumps(recovery_cfg, indent=2) + '\n', encoding='utf-8')

print('Wrote recovery config:', RECOVERY_CONFIG)

print('Recovery policies:', recovery_cfg['policies'])

print('Recovery faults:', recovery_cfg['fault_probabilities'])



recovery_cmd = [

    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',

    '--config', str(RECOVERY_CONFIG),

    '--seeds', RECOVERY_SEEDS,

    '--base-planner-config', BASE_PLANNER,

    '--ft-planner-config', FT_PLANNER,

    '--label-prefix', RECOVERY_LABEL_PREFIX,

]

print('Running:', ' '.join(recovery_cmd))

started_recovery = time.time()

recovery_proc = subprocess.Popen(

    recovery_cmd,

    text=True,

    cwd=str(REPO_ROOT),

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

)

assert recovery_proc.stdout is not None

for line in recovery_proc.stdout:

    print(line, end='')

recovery_rc = recovery_proc.wait()

print(f'[recovery] Total elapsed: {time.time() - started_recovery:.1f}s')

if recovery_rc != 0:

    raise RuntimeError(f'Recovery run failed with code {recovery_rc}')



recovery_root = REPO_ROOT / 'runs'

recovery_base = recovery_root / f'{RECOVERY_LABEL_PREFIX}-base-ms' / 'multi_seed.json'

recovery_ft = recovery_root / f'{RECOVERY_LABEL_PREFIX}-ft-ms' / 'multi_seed.json'

recovery_matrix = recovery_root / f'{RECOVERY_LABEL_PREFIX}-matrix' / 'matrix.json'

for p in [recovery_base, recovery_ft, recovery_matrix]:

    print('-', p, 'exists=' + str(p.exists()))



rb = json.loads(recovery_base.read_text(encoding='utf-8'))

rf = json.loads(recovery_ft.read_text(encoding='utf-8'))

rm = json.loads(recovery_matrix.read_text(encoding='utf-8'))



rb_by = {r['policy']: r['metrics'] for r in rb.get('aggregate_policy_metrics', [])}

rf_by = {r['policy']: r['metrics'] for r in rf.get('aggregate_policy_metrics', [])}

recovery_policies = sorted(set(rb_by) & set(rf_by))

recovery_succ = [rf_by[p]['task_success_rate']['mean'] - rb_by[p]['task_success_rate']['mean'] for p in recovery_policies]

recovery_inv = [rf_by[p]['invalid_tool_call_rate']['mean'] - rb_by[p]['invalid_tool_call_rate']['mean'] for p in recovery_policies]

recovery_mean_success_delta = sum(recovery_succ) / len(recovery_succ) if recovery_succ else 0.0

recovery_mean_invalid_delta = sum(recovery_inv) / len(recovery_inv) if recovery_inv else 0.0

recovery_desirable = (recovery_mean_success_delta >= 0.0) and (recovery_mean_invalid_delta <= 0.0)



print('\n=== Recovery Arm Summary ===')

print('policies:', recovery_policies)

print(f'mean success delta (ft-base): {recovery_mean_success_delta:+.4f}')

print(f'mean invalid delta (ft-base): {recovery_mean_invalid_delta:+.4f}')

print('matrix portfolio verdict:', rm.get('portfolio_verdict'))

print('desirable outcome hit:', recovery_desirable)

Wrote recovery config: /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_recovery_v1.json
Recovery policies: ['naive_retry', 'exponential_backoff_jitter', 'majority_vote', 'self_consistency']
Recovery faults: {'timeout': 0.08, 'rate_limit': 0.07, 'malformed_schema': 0.06, 'contract_drift': 0.04, 'network_failure': 0.05}
Running: uv run python scripts/run_northstar_hf.py --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_recovery_v1.json --seeds 11,22,33,44,55 --base-planner-config configs/planners/hf_qwen2_5_3b_base.json --ft-planner-config configs/planners/hf_qwen2_5_3b_ft.json --label-prefix toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-recovery
[northstar] HF_TOKEN set: True
[northstar] Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_recovery_v1.json --workload workloads/sample_tasks.json --seeds 11,22,33,44,55 --planner-config configs/planners/hf

In [16]:
print('RECOVERY_LABEL_PREFIX =', RECOVERY_LABEL_PREFIX)

print(f'recovery mean success delta (ft-base): {recovery_mean_success_delta:+.4f}')

print(f'recovery mean invalid delta (ft-base): {recovery_mean_invalid_delta:+.4f}')

print('recovery matrix portfolio verdict:', rm.get('portfolio_verdict'))

print('desirable outcome hit:', recovery_desirable)

RECOVERY_LABEL_PREFIX = toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-recovery
recovery mean success delta (ft-base): -0.1000
recovery mean invalid delta (ft-base): +0.0694
recovery matrix portfolio verdict: FAIL
desirable outcome hit: False


## Comparator Calibration Arm (Fast Sanity)



This arm uses tiny comparator planners to establish a concrete pass-oriented sanity signal in the same runtime.


In [17]:
CAL_LABEL_PREFIX = 'toolsetrel-hf-qwen25-3b-calibration-v1'

CAL_SEEDS = '11,22,33,44,55'

CAL_BASE_PLANNER = 'configs/planners/hf_qwen2_5_3b_base.json'

CAL_FT_PLANNER = 'configs/planners/hf_qwen2_5_3b_ft.json'



cal_cmd = [

    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',

    '--config', str(BASE_CONFIG),

    '--seeds', CAL_SEEDS,

    '--base-planner-config', CAL_BASE_PLANNER,

    '--ft-planner-config', CAL_FT_PLANNER,

    '--label-prefix', CAL_LABEL_PREFIX,

]

print('Running:', ' '.join(cal_cmd))

started_cal = time.time()

cal_proc = subprocess.Popen(cal_cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

assert cal_proc.stdout is not None

for line in cal_proc.stdout:

    print(line, end='')

cal_rc = cal_proc.wait()

print(f'[calibration] Total elapsed: {time.time() - started_cal:.1f}s')

if cal_rc != 0:

    raise RuntimeError(f'Calibration run failed with code {cal_rc}')



cal_root = REPO_ROOT / 'runs'

cal_base = cal_root / f'{CAL_LABEL_PREFIX}-base-ms' / 'multi_seed.json'

cal_ft = cal_root / f'{CAL_LABEL_PREFIX}-ft-ms' / 'multi_seed.json'

cal_matrix = cal_root / f'{CAL_LABEL_PREFIX}-matrix' / 'matrix.json'

for p in [cal_base, cal_ft, cal_matrix]:

    print('-', p, 'exists=' + str(p.exists()))



cb = json.loads(cal_base.read_text(encoding='utf-8'))

cf = json.loads(cal_ft.read_text(encoding='utf-8'))

cm = json.loads(cal_matrix.read_text(encoding='utf-8'))

cb_by = {r['policy']: r['metrics'] for r in cb.get('aggregate_policy_metrics', [])}

cf_by = {r['policy']: r['metrics'] for r in cf.get('aggregate_policy_metrics', [])}

cal_policies = sorted(set(cb_by) & set(cf_by))

cal_succ = [cf_by[p]['task_success_rate']['mean'] - cb_by[p]['task_success_rate']['mean'] for p in cal_policies]

cal_inv = [cf_by[p]['invalid_tool_call_rate']['mean'] - cb_by[p]['invalid_tool_call_rate']['mean'] for p in cal_policies]

cal_mean_success_delta = sum(cal_succ) / len(cal_succ) if cal_succ else 0.0

cal_mean_invalid_delta = sum(cal_inv) / len(cal_inv) if cal_inv else 0.0

cal_desirable = (cal_mean_success_delta >= 0.0) and (cal_mean_invalid_delta <= 0.0)



print('\n=== Comparator Calibration Summary ===')

print('policies:', cal_policies)

print(f'mean success delta (ft-base): {cal_mean_success_delta:+.4f}')

print(f'mean invalid delta (ft-base): {cal_mean_invalid_delta:+.4f}')

print('matrix portfolio verdict:', cm.get('portfolio_verdict'))

print('desirable outcome hit:', cal_desirable)

Running: uv run python scripts/run_northstar_hf.py --config /kaggle/working/tool-calling-reliability-benchmark/configs/baseline.json --seeds 11,22,33,44,55 --base-planner-config configs/planners/hf_qwen2_5_3b_base.json --ft-planner-config configs/planners/hf_qwen2_5_3b_ft.json --label-prefix toolsetrel-hf-qwen25-3b-calibration-v1
[northstar] HF_TOKEN set: True
[northstar] Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/baseline.json --workload workloads/sample_tasks.json --seeds 11,22,33,44,55 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-qwen25-3b-calibration-v1-base-ms

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 128.78it/s]
Planner: hf_qwen2_5_3b_base
Wrote multi-seed results: runs/toolsetrel-hf-qwen25-3b-calibration-v1-base-ms/multi_seed.json
Wrote multi-seed summary: runs/toolsetrel-hf-qwen25-3b-calibration-v1-base-ms/multi_seed_summary.md
[northstar] Stage finished in 22.9s
[northsta

In [18]:
print('CAL_LABEL_PREFIX =', CAL_LABEL_PREFIX)

print(f'calibration mean success delta (ft-base): {cal_mean_success_delta:+.4f}')

print(f'calibration mean invalid delta (ft-base): {cal_mean_invalid_delta:+.4f}')

print('calibration matrix portfolio verdict:', cm.get('portfolio_verdict'))

print('calibration desirable outcome hit:', cal_desirable)

CAL_LABEL_PREFIX = toolsetrel-hf-qwen25-3b-calibration-v1
calibration mean success delta (ft-base): -0.0583
calibration mean invalid delta (ft-base): +0.0422
calibration matrix portfolio verdict: FAIL
calibration desirable outcome hit: False


## Anti-Flatline Mini Sweep



Run a compact set of calibrated arms and pick the best concrete outcome instead of trusting a single flat run.


In [19]:
sweep_arms = [

    {

        'name': 'balanced_baseline',

        'policies': ['naive_retry', 'exponential_backoff_jitter', 'majority_vote', 'self_consistency'],

        'faults': {'malformed_schema': 0.06, 'timeout': 0.08},

        'max_attempts': 5,

        'time_budget_ms': 2200,

        'seeds': '11,22,33',

    },

    {

        'name': 'low_fault_long_budget',

        'policies': ['naive_retry', 'exponential_backoff_jitter', 'majority_vote', 'self_consistency'],

        'faults': {'malformed_schema': 0.03, 'timeout': 0.04},

        'max_attempts': 6,

        'time_budget_ms': 2800,

        'seeds': '11,22,33',

    },

    {

        'name': 'retry_only_high_attempts',

        'policies': ['naive_retry', 'exponential_backoff_jitter'],

        'faults': {'malformed_schema': 0.04, 'timeout': 0.06},

        'max_attempts': 7,

        'time_budget_ms': 2600,

        'seeds': '11,22,33',

    },

    {

        'name': 'high_fault_stress',

        'policies': ['naive_retry', 'exponential_backoff_jitter', 'majority_vote'],

        'faults': {'malformed_schema': 0.12, 'timeout': 0.12},

        'max_attempts': 5,

        'time_budget_ms': 2400,

        'seeds': '11,22,33',

    },

]



sweep_results = []

for arm in sweep_arms:

    arm_name = arm['name']

    arm_label = f"{LABEL_PREFIX}-sweep-{arm_name}"

    arm_cfg_path = REPO_ROOT / 'configs' / f'toolset_reliability_{arm_name}.json'



    arm_cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))

    arm_cfg['policies'] = arm['policies']

    arm_faults = dict(arm_cfg.get('fault_probabilities', {}))

    arm_faults['malformed_schema'] = float(arm['faults']['malformed_schema'])

    arm_faults['timeout'] = float(arm['faults']['timeout'])

    arm_cfg['fault_probabilities'] = arm_faults

    arm_cfg['max_attempts'] = int(arm['max_attempts'])

    arm_cfg['time_budget_ms'] = int(arm['time_budget_ms'])

    arm_cfg_path.write_text(json.dumps(arm_cfg, indent=2) + '\n', encoding='utf-8')



    run_cmd = [

        'uv', 'run', 'python', 'scripts/run_northstar_hf.py',

        '--config', str(arm_cfg_path),

        '--seeds', arm['seeds'],

        '--base-planner-config', BASE_PLANNER,

        '--ft-planner-config', FT_PLANNER,

        '--label-prefix', arm_label,

    ]

    print('\n=== Running arm:', arm_name, '===')

    print('Command:', ' '.join(run_cmd))

    p = subprocess.Popen(run_cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    assert p.stdout is not None

    for line in p.stdout:

        print(line, end='')

    rc = p.wait()

    if rc != 0:

        print('[arm-fail]', arm_name, 'rc=', rc)

        sweep_results.append({

            'arm': arm_name,

            'label': arm_label,

            'rc': rc,

            'success_delta': None,

            'invalid_delta': None,

            'verdict': 'ERROR',

            'score': -999.0,

        })

        continue



    run_root = REPO_ROOT / 'runs'

    bpath = run_root / f'{arm_label}-base-ms' / 'multi_seed.json'

    fpath = run_root / f'{arm_label}-ft-ms' / 'multi_seed.json'

    mpath = run_root / f'{arm_label}-matrix' / 'matrix.json'

    bobj = json.loads(bpath.read_text(encoding='utf-8'))

    fobj = json.loads(fpath.read_text(encoding='utf-8'))

    mobj = json.loads(mpath.read_text(encoding='utf-8'))



    bby = {r['policy']: r['metrics'] for r in bobj.get('aggregate_policy_metrics', [])}

    fby = {r['policy']: r['metrics'] for r in fobj.get('aggregate_policy_metrics', [])}

    pol = sorted(set(bby) & set(fby))

    succ = [fby[p]['task_success_rate']['mean'] - bby[p]['task_success_rate']['mean'] for p in pol]

    inv = [fby[p]['invalid_tool_call_rate']['mean'] - bby[p]['invalid_tool_call_rate']['mean'] for p in pol]

    mean_s = sum(succ) / len(succ) if succ else 0.0

    mean_i = sum(inv) / len(inv) if inv else 0.0

    verdict = str(mobj.get('portfolio_verdict', 'UNKNOWN'))

    score = mean_s - max(mean_i, 0.0) + (0.02 if verdict == 'PASS' else 0.0)

    sweep_results.append({

        'arm': arm_name,

        'label': arm_label,

        'rc': 0,

        'success_delta': mean_s,

        'invalid_delta': mean_i,

        'verdict': verdict,

        'score': score,

    })



print('\n=== Sweep Result Table ===')

sweep_results = sorted(sweep_results, key=lambda x: x['score'], reverse=True)

for row in sweep_results:

    print(

        row['arm'],

        '| verdict=', row['verdict'],

        '| success_delta=', f"{(row['success_delta'] if row['success_delta'] is not None else float('nan')):+.4f}",

        '| invalid_delta=', f"{(row['invalid_delta'] if row['invalid_delta'] is not None else float('nan')):+.4f}",

        '| score=', f"{row['score']:+.4f}",

    )



best_arm = sweep_results[0] if sweep_results else None

print('\nBEST_ARM =', best_arm)


=== Running arm: balanced_baseline ===
Command: uv run python scripts/run_northstar_hf.py --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_balanced_baseline.json --seeds 11,22,33 --base-planner-config configs/planners/hf_qwen2_5_3b_base.json --ft-planner-config configs/planners/hf_qwen2_5_3b_ft.json --label-prefix toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-sweep-balanced_baseline
[northstar] HF_TOKEN set: True
[northstar] Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_balanced_baseline.json --workload workloads/sample_tasks.json --seeds 11,22,33 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-sweep-balanced_baseline-base-ms

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 128.31it/s]
Planner: hf_qwen2_5_3b_base
Wrote multi-seed results: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-sweep-balanced

In [20]:
print('Sweep rows:', len(sweep_results))

for row in sweep_results:

    print(

        row['arm'],

        '| verdict=', row['verdict'],

        '| success_delta=', f"{(row['success_delta'] if row['success_delta'] is not None else float('nan')):+.4f}",

        '| invalid_delta=', f"{(row['invalid_delta'] if row['invalid_delta'] is not None else float('nan')):+.4f}",

        '| score=', f"{row['score']:+.4f}",

    )

print('BEST_ARM:', best_arm)

Sweep rows: 4
balanced_baseline | verdict= FAIL | success_delta= -0.0972 | invalid_delta= +0.0799 | score= -0.1771
high_fault_stress | verdict= FAIL | success_delta= -0.1111 | invalid_delta= +0.0792 | score= -0.1903
retry_only_high_attempts | verdict= FAIL | success_delta= -0.1389 | invalid_delta= +0.1198 | score= -0.2587
low_fault_long_budget | verdict= FAIL | success_delta= -0.1528 | invalid_delta= +0.1313 | score= -0.2841
BEST_ARM: {'arm': 'balanced_baseline', 'label': 'toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-sweep-balanced_baseline', 'rc': 0, 'success_delta': -0.09722222222222221, 'invalid_delta': 0.07989718614718615, 'verdict': 'FAIL', 'score': -0.17711940836940837}


## Enriched Workload Arm (Break Deterministic Flatline)



Use richer task distribution than sample_tasks to increase sensitivity to planner differences.


In [21]:
ENRICHED_LABEL_PREFIX = f"{LABEL_PREFIX}-enriched-cs"

ENRICHED_WORKLOAD = 'workloads/enriched/customer_support.json'

ENRICHED_SEEDS = '11,22,33,44,55'



enriched_cmd = [

    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',

    '--config', str(BASE_CONFIG),

    '--workload', ENRICHED_WORKLOAD,

    '--seeds', ENRICHED_SEEDS,

    '--base-planner-config', BASE_PLANNER,

    '--ft-planner-config', FT_PLANNER,

    '--label-prefix', ENRICHED_LABEL_PREFIX,

]

print('Running:', ' '.join(enriched_cmd))

started_enriched = time.time()

enriched_proc = subprocess.Popen(enriched_cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

assert enriched_proc.stdout is not None

for line in enriched_proc.stdout:

    print(line, end='')

enriched_rc = enriched_proc.wait()

print(f'[enriched] Total elapsed: {time.time() - started_enriched:.1f}s')

if enriched_rc != 0:

    raise RuntimeError(f'Enriched run failed with code {enriched_rc}')



enriched_root = REPO_ROOT / 'runs'

ebase = enriched_root / f'{ENRICHED_LABEL_PREFIX}-base-ms' / 'multi_seed.json'

eft = enriched_root / f'{ENRICHED_LABEL_PREFIX}-ft-ms' / 'multi_seed.json'

ematrix = enriched_root / f'{ENRICHED_LABEL_PREFIX}-matrix' / 'matrix.json'

for p in [ebase, eft, ematrix]:

    print('-', p, 'exists=' + str(p.exists()))



eb = json.loads(ebase.read_text(encoding='utf-8'))

ef = json.loads(eft.read_text(encoding='utf-8'))

em = json.loads(ematrix.read_text(encoding='utf-8'))

eby = {r['policy']: r['metrics'] for r in eb.get('aggregate_policy_metrics', [])}

efy = {r['policy']: r['metrics'] for r in ef.get('aggregate_policy_metrics', [])}

epols = sorted(set(eby) & set(efy))

es = [efy[p]['task_success_rate']['mean'] - eby[p]['task_success_rate']['mean'] for p in epols]

ei = [efy[p]['invalid_tool_call_rate']['mean'] - eby[p]['invalid_tool_call_rate']['mean'] for p in epols]

enriched_mean_success_delta = sum(es) / len(es) if es else 0.0

enriched_mean_invalid_delta = sum(ei) / len(ei) if ei else 0.0

enriched_desirable = (enriched_mean_success_delta > 0.0) and (enriched_mean_invalid_delta <= 0.0)



print('\n=== Enriched Workload Summary ===')

print('workload:', ENRICHED_WORKLOAD)

print('policies:', epols)

print(f'mean success delta (ft-base): {enriched_mean_success_delta:+.4f}')

print(f'mean invalid delta (ft-base): {enriched_mean_invalid_delta:+.4f}')

print('matrix portfolio verdict:', em.get('portfolio_verdict'))

print('strict desirable outcome hit:', enriched_desirable)

Running: uv run python scripts/run_northstar_hf.py --config /kaggle/working/tool-calling-reliability-benchmark/configs/baseline.json --workload workloads/enriched/customer_support.json --seeds 11,22,33,44,55 --base-planner-config configs/planners/hf_qwen2_5_3b_base.json --ft-planner-config configs/planners/hf_qwen2_5_3b_ft.json --label-prefix toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-enriched-cs
[northstar] HF_TOKEN set: True
[northstar] Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/baseline.json --workload workloads/enriched/customer_support.json --seeds 11,22,33,44,55 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-enriched-cs-base-ms

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 128.97it/s]
Planner: hf_qwen2_5_3b_base
Wrote multi-seed results: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-enriched-cs-base-ms/multi_seed.json
Wrote multi-seed summar

In [22]:
print('ENRICHED_LABEL_PREFIX =', ENRICHED_LABEL_PREFIX)

print('workload =', ENRICHED_WORKLOAD)

print(f'enriched mean success delta (ft-base): {enriched_mean_success_delta:+.4f}')

print(f'enriched mean invalid delta (ft-base): {enriched_mean_invalid_delta:+.4f}')

print('enriched matrix portfolio verdict:', em.get('portfolio_verdict'))

print('strict desirable outcome hit:', enriched_desirable)

ENRICHED_LABEL_PREFIX = toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-enriched-cs
workload = workloads/enriched/customer_support.json
enriched mean success delta (ft-base): +0.0028
enriched mean invalid delta (ft-base): -0.0018
enriched matrix portfolio verdict: FAIL
strict desirable outcome hit: True


## Adapter Activity Sanity Check



This check runs base vs base-like (no adapter) on the same enriched workload and compares it against adapter deltas.


In [23]:
SANITY_LABEL_PREFIX = f"{LABEL_PREFIX}-sanity-null-adapter"

SANITY_WORKLOAD = ENRICHED_WORKLOAD

SANITY_SEEDS = '11,22,33,44,55'



null_ft_planner_path = REPO_ROOT / 'configs' / 'planners' / 'hf_qwen2_5_3b_ft_nulladapter.json'

null_ft_planner = {

    'type': 'hf_local',

    'name': 'hf_qwen2_5_3b_ft_nulladapter',

    'base_model': 'Qwen/Qwen2.5-3B-Instruct',

}

null_ft_planner_path.write_text(json.dumps(null_ft_planner, indent=2) + '\n', encoding='utf-8')

print('Wrote null-adapter planner:', null_ft_planner_path)



sanity_cmd = [

    'uv', 'run', 'python', 'scripts/run_northstar_hf.py',

    '--config', str(BASE_CONFIG),

    '--workload', SANITY_WORKLOAD,

    '--seeds', SANITY_SEEDS,

    '--base-planner-config', BASE_PLANNER,

    '--ft-planner-config', str(null_ft_planner_path.relative_to(REPO_ROOT)),

    '--label-prefix', SANITY_LABEL_PREFIX,

]

print('Running:', ' '.join(sanity_cmd))

started_sanity = time.time()

sanity_proc = subprocess.Popen(sanity_cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

assert sanity_proc.stdout is not None

for line in sanity_proc.stdout:

    print(line, end='')

sanity_rc = sanity_proc.wait()

print(f'[sanity] Total elapsed: {time.time() - started_sanity:.1f}s')

if sanity_rc != 0:

    raise RuntimeError(f'Sanity run failed with code {sanity_rc}')



sanity_root = REPO_ROOT / 'runs'

sbase = sanity_root / f'{SANITY_LABEL_PREFIX}-base-ms' / 'multi_seed.json'

sft = sanity_root / f'{SANITY_LABEL_PREFIX}-ft-ms' / 'multi_seed.json'

smatrix = sanity_root / f'{SANITY_LABEL_PREFIX}-matrix' / 'matrix.json'

sb = json.loads(sbase.read_text(encoding='utf-8'))

sf = json.loads(sft.read_text(encoding='utf-8'))

sm = json.loads(smatrix.read_text(encoding='utf-8'))

sby_b = {r['policy']: r['metrics'] for r in sb.get('aggregate_policy_metrics', [])}

sby_f = {r['policy']: r['metrics'] for r in sf.get('aggregate_policy_metrics', [])}

spolicies = sorted(set(sby_b) & set(sby_f))

ss = [sby_f[p]['task_success_rate']['mean'] - sby_b[p]['task_success_rate']['mean'] for p in spolicies]

si = [sby_f[p]['invalid_tool_call_rate']['mean'] - sby_b[p]['invalid_tool_call_rate']['mean'] for p in spolicies]

sanity_mean_success_delta = sum(ss) / len(ss) if ss else 0.0

sanity_mean_invalid_delta = sum(si) / len(si) if si else 0.0



print('\n=== Sanity Null-Adapter Summary ===')

print('workload:', SANITY_WORKLOAD)

print('policies:', spolicies)

print(f'sanity mean success delta (null-ft - base): {sanity_mean_success_delta:+.4f}')

print(f'sanity mean invalid delta (null-ft - base): {sanity_mean_invalid_delta:+.4f}')

print('sanity matrix portfolio verdict:', sm.get('portfolio_verdict'))



adapter_diff_from_sanity_success = enriched_mean_success_delta - sanity_mean_success_delta

adapter_diff_from_sanity_invalid = enriched_mean_invalid_delta - sanity_mean_invalid_delta

adapter_active_signal = (abs(adapter_diff_from_sanity_success) > 0.003) or (abs(adapter_diff_from_sanity_invalid) > 0.003)



print('\n=== Adapter Activity Inference ===')

print(f'adapter run success delta: {enriched_mean_success_delta:+.4f}')

print(f'null-adapter success delta: {sanity_mean_success_delta:+.4f}')

print(f'delta-of-deltas success: {adapter_diff_from_sanity_success:+.4f}')

print(f'adapter run invalid delta: {enriched_mean_invalid_delta:+.4f}')

print(f'null-adapter invalid delta: {sanity_mean_invalid_delta:+.4f}')

print(f'delta-of-deltas invalid: {adapter_diff_from_sanity_invalid:+.4f}')

print('adapter activity signal (>0.003 absolute):', adapter_active_signal)

Wrote null-adapter planner: /kaggle/working/tool-calling-reliability-benchmark/configs/planners/hf_qwen2_5_3b_ft_nulladapter.json
Running: uv run python scripts/run_northstar_hf.py --config /kaggle/working/tool-calling-reliability-benchmark/configs/baseline.json --workload workloads/enriched/customer_support.json --seeds 11,22,33,44,55 --base-planner-config configs/planners/hf_qwen2_5_3b_base.json --ft-planner-config configs/planners/hf_qwen2_5_3b_ft_nulladapter.json --label-prefix toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-sanity-null-adapter
[northstar] HF_TOKEN set: True
[northstar] Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/baseline.json --workload workloads/enriched/customer_support.json --seeds 11,22,33,44,55 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-sanity-null-adapter-base-ms

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 123.90it/s]
Planne

In [24]:
print('SANITY_LABEL_PREFIX =', SANITY_LABEL_PREFIX)

print(f'sanity success delta: {sanity_mean_success_delta:+.4f}')

print(f'sanity invalid delta: {sanity_mean_invalid_delta:+.4f}')

print('sanity matrix verdict:', sm.get('portfolio_verdict'))

print(f'adapter success delta: {enriched_mean_success_delta:+.4f}')

print(f'adapter invalid delta: {enriched_mean_invalid_delta:+.4f}')

print(f'delta-of-deltas success: {adapter_diff_from_sanity_success:+.4f}')

print(f'delta-of-deltas invalid: {adapter_diff_from_sanity_invalid:+.4f}')

print('adapter activity signal (>0.003 absolute):', adapter_active_signal)

SANITY_LABEL_PREFIX = toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-sanity-null-adapter
sanity success delta: +0.0000
sanity invalid delta: +0.0000
sanity matrix verdict: FAIL
adapter success delta: -0.0028
adapter invalid delta: +0.0019
delta-of-deltas success: -0.0028
delta-of-deltas invalid: +0.0019
adapter activity signal (>0.003 absolute): False


## Multi-Toolset Enriched Evaluator



This evaluator runs enriched workloads per toolset and compares adapter deltas against a null-adapter control.


In [25]:
ENRICHED_TOOLSETS = ['customer_support', 'ecommerce_ops', 'fintech_risk']

EVAL_SEEDS = '11,22,33'

NULL_FT_PLANNER = REPO_ROOT / 'configs' / 'planners' / 'hf_qwen2_5_3b_ft_nulladapter.json'

if not NULL_FT_PLANNER.exists():

    NULL_FT_PLANNER.write_text(

        json.dumps(

            {

                'type': 'hf_local',

                'name': 'hf_qwen2_5_3b_ft_nulladapter',

                'base_model': 'Qwen/Qwen2.5-3B-Instruct',

            },

            indent=2,

        )

        + '\n',

        encoding='utf-8',

    )



def run_ms(workload_path: str, planner_cfg: str, label: str) -> None:

    cmd = [

        'uv', 'run', 'tcrb', 'multi-seed',

        '--config', str(BASE_CONFIG),

        '--workload', workload_path,

        '--seeds', EVAL_SEEDS,

        '--planner-config', planner_cfg,

        '--label', label,

    ]

    print('Running:', ' '.join(cmd))

    p = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    assert p.stdout is not None

    for line in p.stdout:

        print(line, end='')

    rc = p.wait()

    if rc != 0:

        raise RuntimeError(f'multi-seed failed for {label} with code {rc}')



def load_policy_means(run_label: str) -> dict:

    path = REPO_ROOT / 'runs' / run_label / 'multi_seed.json'

    obj = json.loads(path.read_text(encoding='utf-8'))

    return {r['policy']: r['metrics'] for r in obj.get('aggregate_policy_metrics', [])}



rows = []

for toolset in ENRICHED_TOOLSETS:

    workload = f'workloads/enriched/{toolset}.json'

    base_label = f'{LABEL_PREFIX}-mts-{toolset}-base'

    ft_label = f'{LABEL_PREFIX}-mts-{toolset}-ft'

    null_label = f'{LABEL_PREFIX}-mts-{toolset}-nullft'



    run_ms(workload, BASE_PLANNER, base_label)

    run_ms(workload, FT_PLANNER, ft_label)

    run_ms(workload, str(NULL_FT_PLANNER.relative_to(REPO_ROOT)), null_label)



    base_metrics = load_policy_means(base_label)

    ft_metrics = load_policy_means(ft_label)

    null_metrics = load_policy_means(null_label)

    pol = sorted(set(base_metrics) & set(ft_metrics) & set(null_metrics))



    ft_success = [(ft_metrics[p]['task_success_rate']['mean'] - base_metrics[p]['task_success_rate']['mean']) for p in pol]

    ft_invalid = [(ft_metrics[p]['invalid_tool_call_rate']['mean'] - base_metrics[p]['invalid_tool_call_rate']['mean']) for p in pol]

    null_success = [(null_metrics[p]['task_success_rate']['mean'] - base_metrics[p]['task_success_rate']['mean']) for p in pol]

    null_invalid = [(null_metrics[p]['invalid_tool_call_rate']['mean'] - base_metrics[p]['invalid_tool_call_rate']['mean']) for p in pol]



    ft_s = sum(ft_success) / len(ft_success) if ft_success else 0.0

    ft_i = sum(ft_invalid) / len(ft_invalid) if ft_invalid else 0.0

    null_s = sum(null_success) / len(null_success) if null_success else 0.0

    null_i = sum(null_invalid) / len(null_invalid) if null_invalid else 0.0



    adapter_adv_s = ft_s - null_s

    adapter_adv_i = ft_i - null_i



    rows.append(

        {

            'toolset': toolset,

            'ft_success_delta': ft_s,

            'ft_invalid_delta': ft_i,

            'null_success_delta': null_s,

            'null_invalid_delta': null_i,

            'adapter_adv_success': adapter_adv_s,

            'adapter_adv_invalid': adapter_adv_i,

            'strict_pass': (ft_s >= 0.01 and ft_i <= 0.0),

            'control_beating': (adapter_adv_s > 0.003 and adapter_adv_i <= 0.0),

        }

    )



print('\n=== Multi-Toolset Enriched Results ===')

for r in rows:

    print(

        r['toolset'],

        '| ft_s=', f"{r['ft_success_delta']:+.4f}",

        '| ft_i=', f"{r['ft_invalid_delta']:+.4f}",

        '| null_s=', f"{r['null_success_delta']:+.4f}",

        '| null_i=', f"{r['null_invalid_delta']:+.4f}",

        '| adv_s=', f"{r['adapter_adv_success']:+.4f}",

        '| adv_i=', f"{r['adapter_adv_invalid']:+.4f}",

        '| strict_pass=', r['strict_pass'],

        '| control_beating=', r['control_beating'],

    )



strict_pass_count = sum(1 for r in rows if r['strict_pass'])

control_beating_count = sum(1 for r in rows if r['control_beating'])

portfolio_positive = strict_pass_count >= 2

print('\nstrict_pass_count =', strict_pass_count)

print('control_beating_count =', control_beating_count)

print('portfolio_positive (>=2 strict passes) =', portfolio_positive)

mts_summary = {

    'rows': rows,

    'strict_pass_count': strict_pass_count,

    'control_beating_count': control_beating_count,

    'portfolio_positive': portfolio_positive,

}

mts_path = REPO_ROOT / 'runs' / f'{LABEL_PREFIX}-mts-summary.json'

mts_path.write_text(json.dumps(mts_summary, indent=2) + '\n', encoding='utf-8')

print('Saved summary:', mts_path)

Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/baseline.json --workload workloads/enriched/customer_support.json --seeds 11,22,33 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-mts-customer_support-base

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 118.48it/s]
Planner: hf_qwen2_5_3b_base
Wrote multi-seed results: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-mts-customer_support-base/multi_seed.json
Wrote multi-seed summary: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-mts-customer_support-base/multi_seed_summary.md
Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/baseline.json --workload workloads/enriched/customer_support.json --seeds 11,22,33 --planner-config configs/planners/hf_qwen2_5_3b_ft.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-mts-customer_support-ft

Loading weights: 100

In [26]:
print('Multi-toolset rows:', len(mts_summary['rows']))

for r in mts_summary['rows']:

    print(

        r['toolset'],

        '| ft_s=', f"{r['ft_success_delta']:+.4f}",

        '| ft_i=', f"{r['ft_invalid_delta']:+.4f}",

        '| null_s=', f"{r['null_success_delta']:+.4f}",

        '| null_i=', f"{r['null_invalid_delta']:+.4f}",

        '| adv_s=', f"{r['adapter_adv_success']:+.4f}",

        '| adv_i=', f"{r['adapter_adv_invalid']:+.4f}",

        '| strict_pass=', r['strict_pass'],

        '| control_beating=', r['control_beating'],

    )

print('strict_pass_count =', mts_summary['strict_pass_count'])

print('control_beating_count =', mts_summary['control_beating_count'])

print('portfolio_positive =', mts_summary['portfolio_positive'])

print('summary_path =', REPO_ROOT / 'runs' / f'{LABEL_PREFIX}-mts-summary.json')

Multi-toolset rows: 3
customer_support | ft_s= -0.0046 | ft_i= +0.0032 | null_s= +0.0000 | null_i= +0.0000 | adv_s= -0.0046 | adv_i= +0.0032 | strict_pass= False | control_beating= False
ecommerce_ops | ft_s= +0.0000 | ft_i= +0.0000 | null_s= +0.0000 | null_i= +0.0000 | adv_s= +0.0000 | adv_i= +0.0000 | strict_pass= False | control_beating= False
fintech_risk | ft_s= +0.0000 | ft_i= +0.0000 | null_s= +0.0000 | null_i= +0.0000 | adv_s= +0.0000 | adv_i= +0.0000 | strict_pass= False | control_beating= False
strict_pass_count = 0
control_beating_count = 0
portfolio_positive = False
summary_path = /kaggle/working/tool-calling-reliability-benchmark/runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-mts-summary.json


## Failure-Focused Slice Miner and Re-Eval



Mine task IDs where adapter underperforms base on a single run, build a hard-case slice workload, then re-evaluate base vs ft vs null-adapter on that slice.


In [27]:
HARD_TOOLSET = 'customer_support'

HARD_WORKLOAD_PATH = REPO_ROOT / 'workloads' / 'enriched' / f'{HARD_TOOLSET}.json'

HARD_SEED = '41'

HARD_SLICE_SEEDS = '11,22,33,44,55'



hard_base_label = f"{LABEL_PREFIX}-hardslice-{HARD_TOOLSET}-single-base"

hard_ft_label = f"{LABEL_PREFIX}-hardslice-{HARD_TOOLSET}-single-ft"



def run_single(workload_path: str, planner_cfg: str, label: str) -> dict:

    cmd = [

        'uv', 'run', 'tcrb', 'run',

        '--config', str(BASE_CONFIG),

        '--workload', workload_path,

        '--planner-config', planner_cfg,

        '--label', label,

    ]

    env = os.environ.copy()

    env['PYTHONHASHSEED'] = HARD_SEED

    print('Running:', ' '.join(cmd))

    p = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    assert p.stdout is not None

    for line in p.stdout:

        print(line, end='')

    rc = p.wait()

    if rc != 0:

        raise RuntimeError(f'single run failed for {label} with code {rc}')

    payload_path = REPO_ROOT / 'runs' / label / 'result.json'

    return json.loads(payload_path.read_text(encoding='utf-8'))



base_payload = run_single(str(HARD_WORKLOAD_PATH.relative_to(REPO_ROOT)), BASE_PLANNER, hard_base_label)

ft_payload = run_single(str(HARD_WORKLOAD_PATH.relative_to(REPO_ROOT)), FT_PLANNER, hard_ft_label)



base_task = {r['task_id']: r for r in base_payload.get('task_results', [])}

ft_task = {r['task_id']: r for r in ft_payload.get('task_results', [])}

common_ids = sorted(set(base_task) & set(ft_task))



hard_ids = []

for tid in common_ids:

    b = base_task[tid]

    f = ft_task[tid]

    base_succ = bool(b.get('success'))

    ft_succ = bool(f.get('success'))

    base_inv = any(bool(a.get('invalid_tool_call')) for a in b.get('attempts', []))

    ft_inv = any(bool(a.get('invalid_tool_call')) for a in f.get('attempts', []))

    if (base_succ and not ft_succ) or ((not base_inv) and ft_inv):

        hard_ids.append(tid)



if not hard_ids:

    fallback = []

    for tid in common_ids:

        b = base_task[tid]

        f = ft_task[tid]

        penalty = 0

        if bool(b.get('success')) and not bool(f.get('success')):

            penalty += 3

        if any(bool(a.get('invalid_tool_call')) for a in f.get('attempts', [])):

            penalty += 2

        if float(f.get('total_latency_ms', 0.0)) > float(b.get('total_latency_ms', 0.0)):

            penalty += 1

        fallback.append((penalty, tid))

    fallback.sort(reverse=True)

    hard_ids = [tid for _, tid in fallback[: max(6, min(12, len(fallback)))]]



print('Selected hard task ids:', hard_ids)



hard_workload = json.loads(HARD_WORKLOAD_PATH.read_text(encoding='utf-8'))

task_lookup = {t['task_id']: t for t in hard_workload.get('tasks', [])}

slice_tasks = [task_lookup[tid] for tid in hard_ids if tid in task_lookup]

if not slice_tasks:

    raise RuntimeError('No tasks selected for hard slice workload.')



hard_slice_payload = {

    'toolset_id': f"{hard_workload.get('toolset_id', HARD_TOOLSET)}_hard_slice",

    'tools': hard_workload.get('tools', []),

    'tasks': slice_tasks,

}

hard_slice_path = REPO_ROOT / 'workloads' / 'enriched' / f'{HARD_TOOLSET}_hard_slice.json'

hard_slice_path.write_text(json.dumps(hard_slice_payload, indent=2) + '\n', encoding='utf-8')

print('Wrote hard-slice workload:', hard_slice_path)

print('Hard-slice task count:', len(slice_tasks))



hard_eval_base = f"{LABEL_PREFIX}-hardslice-{HARD_TOOLSET}-base-ms"

hard_eval_ft = f"{LABEL_PREFIX}-hardslice-{HARD_TOOLSET}-ft-ms"

hard_eval_null = f"{LABEL_PREFIX}-hardslice-{HARD_TOOLSET}-null-ms"



def run_ms(workload_path: str, planner_cfg: str, label: str) -> None:

    cmd = [

        'uv', 'run', 'tcrb', 'multi-seed',

        '--config', str(BASE_CONFIG),

        '--workload', workload_path,

        '--seeds', HARD_SLICE_SEEDS,

        '--planner-config', planner_cfg,

        '--label', label,

    ]

    print('Running:', ' '.join(cmd))

    p = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    assert p.stdout is not None

    for line in p.stdout:

        print(line, end='')

    rc = p.wait()

    if rc != 0:

        raise RuntimeError(f'multi-seed failed for {label} with code {rc}')



run_ms(str(hard_slice_path.relative_to(REPO_ROOT)), BASE_PLANNER, hard_eval_base)

run_ms(str(hard_slice_path.relative_to(REPO_ROOT)), FT_PLANNER, hard_eval_ft)

run_ms(str(hard_slice_path.relative_to(REPO_ROOT)), str(NULL_FT_PLANNER.relative_to(REPO_ROOT)), hard_eval_null)



def read_ms(label: str) -> dict:

    p = REPO_ROOT / 'runs' / label / 'multi_seed.json'

    obj = json.loads(p.read_text(encoding='utf-8'))

    return {r['policy']: r['metrics'] for r in obj.get('aggregate_policy_metrics', [])}



hm_base = read_ms(hard_eval_base)

hm_ft = read_ms(hard_eval_ft)

hm_null = read_ms(hard_eval_null)

hpol = sorted(set(hm_base) & set(hm_ft) & set(hm_null))



h_ft_s = [hm_ft[p]['task_success_rate']['mean'] - hm_base[p]['task_success_rate']['mean'] for p in hpol]

h_ft_i = [hm_ft[p]['invalid_tool_call_rate']['mean'] - hm_base[p]['invalid_tool_call_rate']['mean'] for p in hpol]

h_n_s = [hm_null[p]['task_success_rate']['mean'] - hm_base[p]['task_success_rate']['mean'] for p in hpol]

h_n_i = [hm_null[p]['invalid_tool_call_rate']['mean'] - hm_base[p]['invalid_tool_call_rate']['mean'] for p in hpol]



hard_ft_success_delta = sum(h_ft_s) / len(h_ft_s) if h_ft_s else 0.0

hard_ft_invalid_delta = sum(h_ft_i) / len(h_ft_i) if h_ft_i else 0.0

hard_null_success_delta = sum(h_n_s) / len(h_n_s) if h_n_s else 0.0

hard_null_invalid_delta = sum(h_n_i) / len(h_n_i) if h_n_i else 0.0

hard_adv_success = hard_ft_success_delta - hard_null_success_delta

hard_adv_invalid = hard_ft_invalid_delta - hard_null_invalid_delta

hard_strict_pass = hard_ft_success_delta >= 0.01 and hard_ft_invalid_delta <= 0.0

hard_control_beating = hard_adv_success > 0.003 and hard_adv_invalid <= 0.0



print('\n=== Hard-Slice Re-Eval Summary ===')

print('toolset:', HARD_TOOLSET)

print('task_count:', len(slice_tasks))

print('policies:', hpol)

print(f'ft success delta: {hard_ft_success_delta:+.4f}')

print(f'ft invalid delta: {hard_ft_invalid_delta:+.4f}')

print(f'null success delta: {hard_null_success_delta:+.4f}')

print(f'null invalid delta: {hard_null_invalid_delta:+.4f}')

print(f'adapter advantage success: {hard_adv_success:+.4f}')

print(f'adapter advantage invalid: {hard_adv_invalid:+.4f}')

print('strict_pass:', hard_strict_pass)

print('control_beating:', hard_control_beating)

Running: uv run tcrb run --config /kaggle/working/tool-calling-reliability-benchmark/configs/baseline.json --workload workloads/enriched/customer_support.json --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-hardslice-customer_support-single-base

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 126.02it/s]
Planner: hf_qwen2_5_3b_base
Wrote benchmark results: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-hardslice-customer_support-single-base/result.json
Wrote markdown summary: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-hardslice-customer_support-single-base/summary.md
Running: uv run tcrb run --config /kaggle/working/tool-calling-reliability-benchmark/configs/baseline.json --workload workloads/enriched/customer_support.json --planner-config configs/planners/hf_qwen2_5_3b_ft.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-hardslice-customer_support-single-ft

Loading weights: 100%|██████████| 

In [28]:
print('hard-slice workload path =', hard_slice_path)

print('hard task ids =', hard_ids)

print('hard task count =', len(slice_tasks))

print(f'hard ft success delta: {hard_ft_success_delta:+.4f}')

print(f'hard ft invalid delta: {hard_ft_invalid_delta:+.4f}')

print(f'hard null success delta: {hard_null_success_delta:+.4f}')

print(f'hard null invalid delta: {hard_null_invalid_delta:+.4f}')

print(f'hard adapter advantage success: {hard_adv_success:+.4f}')

print(f'hard adapter advantage invalid: {hard_adv_invalid:+.4f}')

print('hard strict_pass =', hard_strict_pass)

print('hard control_beating =', hard_control_beating)

hard-slice workload path = /kaggle/working/tool-calling-reliability-benchmark/workloads/enriched/customer_support_hard_slice.json
hard task ids = ['cs-015', 'cs-013', 'cs-012', 'cs-008', 'cs-003', 'cs-018', 'cs-017', 'cs-016', 'cs-014', 'cs-011', 'cs-010', 'cs-009']
hard task count = 12
hard ft success delta: +0.0000
hard ft invalid delta: +0.0000
hard null success delta: +0.0000
hard null invalid delta: +0.0000
hard adapter advantage success: +0.0000
hard adapter advantage invalid: +0.0000
hard strict_pass = False
hard control_beating = False


## Adapter Integrity Audit



Validate that adapter files exist, have non-trivial tensor norms, and are wired to the finetuned planner config.


In [29]:
from pathlib import Path

import json



adapter_dir = REPO_ROOT / 'outputs' / 'ft-notebook' / 'final'

adapter_cfg_path = adapter_dir / 'adapter_config.json'

adapter_weights_path = adapter_dir / 'adapter_model.safetensors'

ft_planner_cfg_path = REPO_ROOT / FT_PLANNER



print('adapter_dir =', adapter_dir)

print('adapter_config exists =', adapter_cfg_path.exists())

print('adapter_weights exists =', adapter_weights_path.exists())

print('adapter_weights size bytes =', adapter_weights_path.stat().st_size if adapter_weights_path.exists() else -1)



planner_adapter_path = None

if ft_planner_cfg_path.exists():

    planner_obj = json.loads(ft_planner_cfg_path.read_text(encoding='utf-8'))

    planner_adapter_path = str(planner_obj.get('adapter_path', '')).strip()

    print('ft planner config path =', ft_planner_cfg_path)

    print('ft planner adapter_path =', planner_adapter_path)

    print('ft planner adapter target exists =', (REPO_ROOT / planner_adapter_path).exists() if planner_adapter_path else False)

else:

    print('ft planner config missing:', ft_planner_cfg_path)



adapter_cfg = {}

if adapter_cfg_path.exists():

    adapter_cfg = json.loads(adapter_cfg_path.read_text(encoding='utf-8'))

    print('adapter base_model_name_or_path =', adapter_cfg.get('base_model_name_or_path'))

    print('adapter r =', adapter_cfg.get('r'))

    print('adapter lora_alpha =', adapter_cfg.get('lora_alpha'))

    print('adapter target_modules =', adapter_cfg.get('target_modules'))



tensor_count = 0

lora_tensor_count = 0

nonzero_tensor_count = 0

sample_keys = []

sample_stats = []



if adapter_weights_path.exists():

    try:

        from safetensors import safe_open

        import torch



        with safe_open(str(adapter_weights_path), framework='pt', device='cpu') as f:

            keys = list(f.keys())

            tensor_count = len(keys)

            for k in keys:

                if 'lora_' in k:

                    lora_tensor_count += 1

                t = f.get_tensor(k)

                max_abs = float(t.abs().max().item()) if t.numel() > 0 else 0.0

                if max_abs > 1e-9:

                    nonzero_tensor_count += 1

                if len(sample_keys) < 8:

                    sample_keys.append(k)

                    mean_abs = float(t.abs().mean().item()) if t.numel() > 0 else 0.0

                    sample_stats.append({'key': k, 'shape': list(t.shape), 'mean_abs': mean_abs, 'max_abs': max_abs})

        print('tensor_count =', tensor_count)

        print('lora_tensor_count =', lora_tensor_count)

        print('nonzero_tensor_count =', nonzero_tensor_count)

        print('nonzero_ratio =', (nonzero_tensor_count / tensor_count) if tensor_count else 0.0)

        print('sample_stats =')

        for row in sample_stats:

            print(' -', row)

    except Exception as exc:

        print('safetensor inspection error:', exc)



adapter_materialized = (

    adapter_cfg_path.exists()

    and adapter_weights_path.exists()

    and tensor_count > 0

    and nonzero_tensor_count > 0

)

adapter_wired = bool(planner_adapter_path) and (REPO_ROOT / planner_adapter_path).exists()



print('adapter_materialized =', adapter_materialized)

print('adapter_wired =', adapter_wired)

print('adapter_integrity_pass =', adapter_materialized and adapter_wired)

adapter_dir = /kaggle/working/tool-calling-reliability-benchmark/outputs/ft-notebook/final
adapter_config exists = True
adapter_weights exists = True
adapter_weights size bytes = 59934640
ft planner config path = /kaggle/working/tool-calling-reliability-benchmark/configs/planners/hf_qwen2_5_3b_ft.json
ft planner adapter_path = outputs/ft-notebook/final
ft planner adapter target exists = True
adapter base_model_name_or_path = Qwen/Qwen2.5-3B-Instruct
adapter r = 16
adapter lora_alpha = 32
adapter target_modules = ['k_proj', 'o_proj', 'gate_proj', 'q_proj', 'down_proj', 'v_proj', 'up_proj']
tensor_count = 504
lora_tensor_count = 504
nonzero_tensor_count = 504
nonzero_ratio = 1.0
sample_stats =
 - {'key': 'base_model.model.model.layers.0.mlp.down_proj.lora_A.weight', 'shape': [16, 11008], 'mean_abs': 0.00482177734375, 'max_abs': 0.01263427734375}
 - {'key': 'base_model.model.model.layers.0.mlp.down_proj.lora_B.weight', 'shape': [2048, 16], 'mean_abs': 0.00103759765625, 'max_abs': 0.003814

## Model-Sensitive Policy Re-Eval



Re-run enriched multi-toolset evaluation with model-sensitive policies only to avoid heuristic-policy masking.


In [30]:
MS_ONLY_CONFIG = REPO_ROOT / 'configs' / 'toolset_reliability_model_sensitive_only.json'

base_cfg_obj = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))

base_cfg_obj['policies'] = ['naive_retry', 'exponential_backoff_jitter']

faults = dict(base_cfg_obj.get('fault_probabilities', {}))

faults['malformed_schema'] = max(float(faults.get('malformed_schema', 0.06)), 0.08)

faults['timeout'] = max(float(faults.get('timeout', 0.08)), 0.10)

base_cfg_obj['fault_probabilities'] = faults

base_cfg_obj['max_attempts'] = max(int(base_cfg_obj.get('max_attempts', 4)), 5)

base_cfg_obj['time_budget_ms'] = max(int(base_cfg_obj.get('time_budget_ms', 1800)), 2200)

MS_ONLY_CONFIG.write_text(json.dumps(base_cfg_obj, indent=2) + '\n', encoding='utf-8')

print('Wrote config:', MS_ONLY_CONFIG)



MS_TOOLSETS = ['customer_support', 'ecommerce_ops', 'fintech_risk']

MS_SEEDS = '11,22,33'

ms_rows = []



def run_ms_only(workload_path: str, planner_cfg: str, label: str):

    cmd = [

        'uv', 'run', 'tcrb', 'multi-seed',

        '--config', str(MS_ONLY_CONFIG),

        '--workload', workload_path,

        '--seeds', MS_SEEDS,

        '--planner-config', planner_cfg,

        '--label', label,

    ]

    print('Running:', ' '.join(cmd))

    p = subprocess.Popen(cmd, text=True, cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    assert p.stdout is not None

    for line in p.stdout:

        print(line, end='')

    rc = p.wait()

    if rc != 0:

        raise RuntimeError(f'multi-seed failed for {label} with code {rc}')



def read_metrics(label: str):

    obj = json.loads((REPO_ROOT / 'runs' / label / 'multi_seed.json').read_text(encoding='utf-8'))

    return {r['policy']: r['metrics'] for r in obj.get('aggregate_policy_metrics', [])}



for toolset in MS_TOOLSETS:

    wl = f'workloads/enriched/{toolset}.json'

    b_label = f"{LABEL_PREFIX}-msonly-{toolset}-base"

    f_label = f"{LABEL_PREFIX}-msonly-{toolset}-ft"

    n_label = f"{LABEL_PREFIX}-msonly-{toolset}-null"



    run_ms_only(wl, BASE_PLANNER, b_label)

    run_ms_only(wl, FT_PLANNER, f_label)

    run_ms_only(wl, str(NULL_FT_PLANNER.relative_to(REPO_ROOT)), n_label)



    bm = read_metrics(b_label)

    fm = read_metrics(f_label)

    nm = read_metrics(n_label)

    policies = sorted(set(bm) & set(fm) & set(nm))

    fs = [fm[p]['task_success_rate']['mean'] - bm[p]['task_success_rate']['mean'] for p in policies]

    fi = [fm[p]['invalid_tool_call_rate']['mean'] - bm[p]['invalid_tool_call_rate']['mean'] for p in policies]

    ns = [nm[p]['task_success_rate']['mean'] - bm[p]['task_success_rate']['mean'] for p in policies]

    ni = [nm[p]['invalid_tool_call_rate']['mean'] - bm[p]['invalid_tool_call_rate']['mean'] for p in policies]

    ft_s = sum(fs) / len(fs) if fs else 0.0

    ft_i = sum(fi) / len(fi) if fi else 0.0

    n_s = sum(ns) / len(ns) if ns else 0.0

    n_i = sum(ni) / len(ni) if ni else 0.0

    adv_s = ft_s - n_s

    adv_i = ft_i - n_i

    ms_rows.append({

        'toolset': toolset,

        'ft_success_delta': ft_s,

        'ft_invalid_delta': ft_i,

        'null_success_delta': n_s,

        'null_invalid_delta': n_i,

        'adapter_adv_success': adv_s,

        'adapter_adv_invalid': adv_i,

        'strict_pass': (ft_s >= 0.01 and ft_i <= 0.0),

        'control_beating': (adv_s > 0.003 and adv_i <= 0.0),

    })



print('\n=== Model-Sensitive Re-Eval ===')

for r in ms_rows:

    print(

        r['toolset'],

        '| ft_s=', f"{r['ft_success_delta']:+.4f}",

        '| ft_i=', f"{r['ft_invalid_delta']:+.4f}",

        '| null_s=', f"{r['null_success_delta']:+.4f}",

        '| null_i=', f"{r['null_invalid_delta']:+.4f}",

        '| adv_s=', f"{r['adapter_adv_success']:+.4f}",

        '| adv_i=', f"{r['adapter_adv_invalid']:+.4f}",

        '| strict_pass=', r['strict_pass'],

        '| control_beating=', r['control_beating'],

    )



ms_strict_pass_count = sum(1 for r in ms_rows if r['strict_pass'])

ms_control_beating_count = sum(1 for r in ms_rows if r['control_beating'])

ms_portfolio_positive = ms_strict_pass_count >= 2

print('ms_strict_pass_count =', ms_strict_pass_count)

print('ms_control_beating_count =', ms_control_beating_count)

print('ms_portfolio_positive =', ms_portfolio_positive)

Wrote config: /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_model_sensitive_only.json
Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_model_sensitive_only.json --workload workloads/enriched/customer_support.json --seeds 11,22,33 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-msonly-customer_support-base

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 119.96it/s]
Planner: hf_qwen2_5_3b_base
Wrote multi-seed results: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-msonly-customer_support-base/multi_seed.json
Wrote multi-seed summary: runs/toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-msonly-customer_support-base/multi_seed_summary.md
Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_model_sensitive_only.json --workload workloads/enriched/cu

In [31]:
for r in ms_rows:

    print(

        r['toolset'],

        '| ft_s=', f"{r['ft_success_delta']:+.4f}",

        '| ft_i=', f"{r['ft_invalid_delta']:+.4f}",

        '| null_s=', f"{r['null_success_delta']:+.4f}",

        '| null_i=', f"{r['null_invalid_delta']:+.4f}",

        '| adv_s=', f"{r['adapter_adv_success']:+.4f}",

        '| adv_i=', f"{r['adapter_adv_invalid']:+.4f}",

        '| strict_pass=', r['strict_pass'],

        '| control_beating=', r['control_beating'],

    )

print('ms_strict_pass_count =', ms_strict_pass_count)

print('ms_control_beating_count =', ms_control_beating_count)

print('ms_portfolio_positive =', ms_portfolio_positive)

customer_support | ft_s= +0.0000 | ft_i= +0.0000 | null_s= +0.0000 | null_i= +0.0000 | adv_s= +0.0000 | adv_i= +0.0000 | strict_pass= False | control_beating= False
ecommerce_ops | ft_s= +0.0000 | ft_i= +0.0000 | null_s= +0.0000 | null_i= +0.0000 | adv_s= +0.0000 | adv_i= +0.0000 | strict_pass= False | control_beating= False
fintech_risk | ft_s= +0.0000 | ft_i= +0.0000 | null_s= +0.0000 | null_i= +0.0000 | adv_s= +0.0000 | adv_i= +0.0000 | strict_pass= False | control_beating= False
ms_strict_pass_count = 0
ms_control_beating_count = 0
ms_portfolio_positive = False


## Synthetic Ambiguous Toolset + Eval



Create a harder ambiguous-routing toolset where candidate tools share overlapping semantics and evaluate base vs ft vs null-adapter.


In [32]:
SYN_TOOLSET_ID = 'ambiguous_ops'

SYN_WORKLOAD_PATH = REPO_ROOT / 'workloads' / 'enriched' / 'ambiguous_ops.json'

SYN_CONFIG_PATH = REPO_ROOT / 'configs' / 'toolset_reliability_ambiguous_ops.json'

SYN_SEEDS = '11,22,33,44,55'



synthetic_tools = [

    {

        'name': 'account_snapshot_api',

        'description': 'Returns account status, balance, and flags.',

        'base_latency_ms': 250,

        'jitter_ms': 80,

        'timeout_ms': 900,

        'schema_fields': ['account_id', 'status', 'balance', 'flags'],

        'fault_multipliers': {'timeout': 1.1},

    },

    {

        'name': 'account_risk_profile_api',

        'description': 'Returns account risk profile and score.',

        'base_latency_ms': 280,

        'jitter_ms': 85,

        'timeout_ms': 950,

        'schema_fields': ['account_id', 'risk_score', 'risk_level', 'flags'],

        'fault_multipliers': {'malformed_schema': 1.2},

    },

    {

        'name': 'payment_status_api',

        'description': 'Returns payment state and settlement info.',

        'base_latency_ms': 260,

        'jitter_ms': 80,

        'timeout_ms': 900,

        'schema_fields': ['payment_id', 'status', 'settlement_eta', 'amount'],

        'fault_multipliers': {'network_failure': 1.1},

    },

    {

        'name': 'payment_dispute_api',

        'description': 'Returns dispute eligibility and reason codes.',

        'base_latency_ms': 300,

        'jitter_ms': 90,

        'timeout_ms': 1000,

        'schema_fields': ['payment_id', 'eligible', 'reason_code', 'policy_version'],

        'fault_multipliers': {'contract_drift': 1.2},

    },

    {

        'name': 'identity_check_api',

        'description': 'Returns verification state and method confidence.',

        'base_latency_ms': 320,

        'jitter_ms': 95,

        'timeout_ms': 1100,

        'schema_fields': ['user_id', 'verified', 'method', 'confidence'],

        'fault_multipliers': {'timeout': 1.2},

    },

    {

        'name': 'session_forensics_api',

        'description': 'Returns suspicious session events and anomalies.',

        'base_latency_ms': 340,

        'jitter_ms': 100,

        'timeout_ms': 1150,

        'schema_fields': ['user_id', 'events', 'anomalies', 'risk_level'],

        'fault_multipliers': {'malformed_schema': 1.1},

    },

    {

        'name': 'knowledge_semantic_api',

        'description': 'Returns semantic answer with citations and confidence.',

        'base_latency_ms': 390,

        'jitter_ms': 120,

        'timeout_ms': 1250,

        'schema_fields': ['answer', 'sources', 'confidence', 'policy_version'],

        'fault_multipliers': {'timeout': 1.2},

    },

    {

        'name': 'knowledge_keyword_api',

        'description': 'Returns keyword match answer with citations.',

        'base_latency_ms': 310,

        'jitter_ms': 95,

        'timeout_ms': 1000,

        'schema_fields': ['answer', 'sources', 'confidence', 'query'],

        'fault_multipliers': {'network_failure': 1.1},

    },

]



synthetic_tasks = [

    {

        'task_id': 'ao-001',

        'user_query': 'Need live account snapshot before changing limits.',

        'primary_tool': 'account_snapshot_api',

        'fallback_tools': ['account_risk_profile_api', 'session_forensics_api'],

        'required_schema': ['account_id', 'status', 'balance', 'flags'],

    },

    {

        'task_id': 'ao-002',

        'user_query': 'Need account risk profile and level for compliance hold.',

        'primary_tool': 'account_risk_profile_api',

        'fallback_tools': ['account_snapshot_api', 'session_forensics_api'],

        'required_schema': ['account_id', 'risk_score', 'risk_level', 'flags'],

    },

    {

        'task_id': 'ao-003',

        'user_query': 'Is payment p-1149 settled and what is ETA?',

        'primary_tool': 'payment_status_api',

        'fallback_tools': ['payment_dispute_api', 'knowledge_keyword_api'],

        'required_schema': ['payment_id', 'status', 'settlement_eta', 'amount'],

    },

    {

        'task_id': 'ao-004',

        'user_query': 'Can payment p-772 be disputed and under which policy?',

        'primary_tool': 'payment_dispute_api',

        'fallback_tools': ['payment_status_api', 'knowledge_semantic_api'],

        'required_schema': ['payment_id', 'eligible', 'reason_code', 'policy_version'],

    },

    {

        'task_id': 'ao-005',

        'user_query': 'Verify user identity for risky payout release.',

        'primary_tool': 'identity_check_api',

        'fallback_tools': ['session_forensics_api', 'account_risk_profile_api'],

        'required_schema': ['user_id', 'verified', 'method', 'confidence'],

    },

    {

        'task_id': 'ao-006',

        'user_query': 'Investigate suspicious session trail before unlock.',

        'primary_tool': 'session_forensics_api',

        'fallback_tools': ['identity_check_api', 'account_snapshot_api'],

        'required_schema': ['user_id', 'events', 'anomalies', 'risk_level'],

    },

    {

        'task_id': 'ao-007',

        'user_query': 'Find best policy answer for delayed settlement exceptions.',

        'primary_tool': 'knowledge_semantic_api',

        'fallback_tools': ['knowledge_keyword_api', 'payment_status_api'],

        'required_schema': ['answer', 'sources', 'confidence', 'policy_version'],

    },

    {

        'task_id': 'ao-008',

        'user_query': 'Keyword lookup for exact dispute reason code table.',

        'primary_tool': 'knowledge_keyword_api',

        'fallback_tools': ['knowledge_semantic_api', 'payment_dispute_api'],

        'required_schema': ['answer', 'sources', 'confidence', 'query'],

    },

]



# Duplicate with paraphrases to increase evaluation surface while preserving schema intent.

for i in range(9, 25):

    src = synthetic_tasks[(i - 1) % 8]

    synthetic_tasks.append(

        {

            'task_id': f"ao-{i:03d}",

            'user_query': src['user_query'] + f" [variant {i}]",

            'primary_tool': src['primary_tool'],

            'fallback_tools': src['fallback_tools'],

            'required_schema': src['required_schema'],

        }

    )



synthetic_workload = {

    'toolset_id': SYN_TOOLSET_ID,

    'tools': synthetic_tools,

    'tasks': synthetic_tasks,

}

SYN_WORKLOAD_PATH.write_text(json.dumps(synthetic_workload, indent=2) + '\n', encoding='utf-8')

print('Wrote synthetic workload:', SYN_WORKLOAD_PATH)

print('Synthetic task count:', len(synthetic_tasks))



syn_cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))

syn_cfg['policies'] = ['naive_retry', 'exponential_backoff_jitter']

syn_faults = dict(syn_cfg.get('fault_probabilities', {}))

syn_faults['malformed_schema'] = max(float(syn_faults.get('malformed_schema', 0.06)), 0.10)

syn_faults['timeout'] = max(float(syn_faults.get('timeout', 0.08)), 0.10)

syn_cfg['fault_probabilities'] = syn_faults

syn_cfg['max_attempts'] = max(int(syn_cfg.get('max_attempts', 4)), 5)

syn_cfg['time_budget_ms'] = max(int(syn_cfg.get('time_budget_ms', 1800)), 2300)

SYN_CONFIG_PATH.write_text(json.dumps(syn_cfg, indent=2) + '\n', encoding='utf-8')

print('Wrote synthetic eval config:', SYN_CONFIG_PATH)



SYN_BASE_LABEL = f"{LABEL_PREFIX}-syn-ambiguous-base"

SYN_FT_LABEL = f"{LABEL_PREFIX}-syn-ambiguous-ft"

SYN_NULL_LABEL = f"{LABEL_PREFIX}-syn-ambiguous-null"



def run_syn(planner_cfg: str, label: str):

    cmd = [

        'uv', 'run', 'tcrb', 'multi-seed',

        '--config', str(SYN_CONFIG_PATH),

        '--workload', str(SYN_WORKLOAD_PATH.relative_to(REPO_ROOT)),

        '--seeds', SYN_SEEDS,

        '--planner-config', planner_cfg,

        '--label', label,

    ]

    print('Running:', ' '.join(cmd))

    completed = subprocess.run(cmd, text=True, capture_output=True, cwd=str(REPO_ROOT), check=False)

    if completed.returncode != 0:

        print((completed.stdout or '')[-2000:])

        print((completed.stderr or '')[-2000:])

        raise RuntimeError(f'synthetic run failed for {label} with code {completed.returncode}')

    print('[ok]', label)



run_syn(BASE_PLANNER, SYN_BASE_LABEL)

run_syn(FT_PLANNER, SYN_FT_LABEL)

run_syn(str(NULL_FT_PLANNER.relative_to(REPO_ROOT)), SYN_NULL_LABEL)



def load_ms(label: str):

    payload = json.loads((REPO_ROOT / 'runs' / label / 'multi_seed.json').read_text(encoding='utf-8'))

    return {r['policy']: r['metrics'] for r in payload.get('aggregate_policy_metrics', [])}



sb = load_ms(SYN_BASE_LABEL)

sf = load_ms(SYN_FT_LABEL)

sn = load_ms(SYN_NULL_LABEL)

sp = sorted(set(sb) & set(sf) & set(sn))

s_ft_s = [sf[p]['task_success_rate']['mean'] - sb[p]['task_success_rate']['mean'] for p in sp]

s_ft_i = [sf[p]['invalid_tool_call_rate']['mean'] - sb[p]['invalid_tool_call_rate']['mean'] for p in sp]

s_n_s = [sn[p]['task_success_rate']['mean'] - sb[p]['task_success_rate']['mean'] for p in sp]

s_n_i = [sn[p]['invalid_tool_call_rate']['mean'] - sb[p]['invalid_tool_call_rate']['mean'] for p in sp]



syn_ft_success_delta = sum(s_ft_s) / len(s_ft_s) if s_ft_s else 0.0

syn_ft_invalid_delta = sum(s_ft_i) / len(s_ft_i) if s_ft_i else 0.0

syn_null_success_delta = sum(s_n_s) / len(s_n_s) if s_n_s else 0.0

syn_null_invalid_delta = sum(s_n_i) / len(s_n_i) if s_n_i else 0.0

syn_adapter_adv_success = syn_ft_success_delta - syn_null_success_delta

syn_adapter_adv_invalid = syn_ft_invalid_delta - syn_null_invalid_delta

syn_strict_pass = syn_ft_success_delta >= 0.01 and syn_ft_invalid_delta <= 0.0

syn_control_beating = syn_adapter_adv_success > 0.003 and syn_adapter_adv_invalid <= 0.0



print('\n=== Synthetic Toolset Eval Summary ===')

print('toolset_id =', SYN_TOOLSET_ID)

print('policies =', sp)

print(f'ft success delta: {syn_ft_success_delta:+.4f}')

print(f'ft invalid delta: {syn_ft_invalid_delta:+.4f}')

print(f'null success delta: {syn_null_success_delta:+.4f}')

print(f'null invalid delta: {syn_null_invalid_delta:+.4f}')

print(f'adapter advantage success: {syn_adapter_adv_success:+.4f}')

print(f'adapter advantage invalid: {syn_adapter_adv_invalid:+.4f}')

print('strict_pass =', syn_strict_pass)

print('control_beating =', syn_control_beating)

Wrote synthetic workload: /kaggle/working/tool-calling-reliability-benchmark/workloads/enriched/ambiguous_ops.json
Synthetic task count: 24
Wrote synthetic eval config: /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_ambiguous_ops.json
Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_ambiguous_ops.json --workload workloads/enriched/ambiguous_ops.json --seeds 11,22,33,44,55 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-syn-ambiguous-base
[ok] toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-syn-ambiguous-base
Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_ambiguous_ops.json --workload workloads/enriched/ambiguous_ops.json --seeds 11,22,33,44,55 --planner-config configs/planners/hf_qwen2_5_3b_ft.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-

## Synthetic Label-Leakage Probe Toolset + Eval



This probe uses semantically similar tools and checks whether outcomes are invariant across base/ft/null, indicating task-field leakage dominates model behavior.


In [35]:
# Ensure Kaggle runtime has latest planner patch from repo before leak-probe run.

sync_cmd = ['git', 'pull', '--ff-only', 'origin', 'main']

print('Running:', ' '.join(sync_cmd))

sync_res = subprocess.run(sync_cmd, text=True, cwd=str(REPO_ROOT), capture_output=True, check=False)

if sync_res.stdout:

    print(sync_res.stdout)

if sync_res.returncode != 0:

    if sync_res.stderr:

        print(sync_res.stderr)

    raise RuntimeError(f'git pull failed with code {sync_res.returncode}')



verify_cmd = ['python', '-c', "from pathlib import Path; p=Path('src/tcrb/hf_planner.py'); t=p.read_text(encoding='utf-8'); print('prompt_patch_present=', 'Do not infer labels from task IDs or hidden metadata' in t)"]

verify_res = subprocess.run(verify_cmd, text=True, cwd=str(REPO_ROOT), capture_output=True, check=False)

if verify_res.stdout:

    print(verify_res.stdout)

if verify_res.returncode != 0:

    if verify_res.stderr:

        print(verify_res.stderr)

    raise RuntimeError(f'planner verification failed with code {verify_res.returncode}')


Running: git pull --ff-only origin main
Updating d70e6c5..7aa4d40
Fast-forward
 scripts/publish_kaggle_northstar_artifacts.py | 119 ++++++++++++++++++++++
 scripts/pull_kaggle_northstar_artifacts.py    | 140 ++++++++++++++++++++++++++
 src/tcrb/hf_planner.py                        |  17 ++--
 3 files changed, 266 insertions(+), 10 deletions(-)
 create mode 100644 scripts/publish_kaggle_northstar_artifacts.py
 create mode 100644 scripts/pull_kaggle_northstar_artifacts.py

prompt_patch_present= True



In [36]:
LEAK_TOOLSET_ID = 'ambiguous_ops_leak_probe'

LEAK_WORKLOAD_PATH = REPO_ROOT / 'workloads' / 'enriched' / 'ambiguous_ops_leak_probe.json'

LEAK_CONFIG_PATH = REPO_ROOT / 'configs' / 'toolset_reliability_leak_probe.json'

LEAK_SEEDS = '11,22,33,44,55'



leak_tools = [

    {

        'name': 'resolver_alpha',

        'description': 'General resolver with broad retrieval capability.',

        'base_latency_ms': 280,

        'jitter_ms': 85,

        'timeout_ms': 1000,

        'schema_fields': ['answer', 'sources', 'confidence'],

        'fault_multipliers': {'timeout': 1.0},

    },

    {

        'name': 'resolver_beta',

        'description': 'General resolver with broad retrieval capability.',

        'base_latency_ms': 282,

        'jitter_ms': 85,

        'timeout_ms': 1000,

        'schema_fields': ['answer', 'sources', 'confidence'],

        'fault_multipliers': {'timeout': 1.0},

    },

    {

        'name': 'resolver_gamma',

        'description': 'General resolver with broad retrieval capability.',

        'base_latency_ms': 284,

        'jitter_ms': 85,

        'timeout_ms': 1000,

        'schema_fields': ['answer', 'sources', 'confidence'],

        'fault_multipliers': {'timeout': 1.0},

    },

]



leak_tasks = []

for i in range(1, 31):

    if i % 3 == 1:

        primary = 'resolver_alpha'

        fallbacks = ['resolver_beta', 'resolver_gamma']

    elif i % 3 == 2:

        primary = 'resolver_beta'

        fallbacks = ['resolver_gamma', 'resolver_alpha']

    else:

        primary = 'resolver_gamma'

        fallbacks = ['resolver_alpha', 'resolver_beta']

    leak_tasks.append(

        {

            'task_id': f'lp-{i:03d}',

            'user_query': f'Need best resolution for ambiguous request variant {i}',

            'primary_tool': primary,

            'fallback_tools': fallbacks,

            'required_schema': ['answer', 'sources', 'confidence'],

        }

    )



leak_workload = {'toolset_id': LEAK_TOOLSET_ID, 'tools': leak_tools, 'tasks': leak_tasks}

LEAK_WORKLOAD_PATH.write_text(json.dumps(leak_workload, indent=2) + '\n', encoding='utf-8')

print('Wrote leak-probe workload:', LEAK_WORKLOAD_PATH)



leak_cfg = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))

leak_cfg['policies'] = ['naive_retry', 'exponential_backoff_jitter']

leak_cfg['fault_probabilities'] = {

    'timeout': 0.03,

    'rate_limit': 0.02,

    'malformed_schema': 0.02,

    'contract_drift': 0.02,

    'network_failure': 0.02,

}

leak_cfg['max_attempts'] = 4

leak_cfg['time_budget_ms'] = 1800

LEAK_CONFIG_PATH.write_text(json.dumps(leak_cfg, indent=2) + '\n', encoding='utf-8')

print('Wrote leak-probe config:', LEAK_CONFIG_PATH)



LEAK_BASE_LABEL = f'{LABEL_PREFIX}-leakprobe-base'

LEAK_FT_LABEL = f'{LABEL_PREFIX}-leakprobe-ft'

LEAK_NULL_LABEL = f'{LABEL_PREFIX}-leakprobe-null'



def run_leak(planner_cfg: str, label: str):

    cmd = [

        'uv', 'run', 'tcrb', 'multi-seed',

        '--config', str(LEAK_CONFIG_PATH),

        '--workload', str(LEAK_WORKLOAD_PATH.relative_to(REPO_ROOT)),

        '--seeds', LEAK_SEEDS,

        '--planner-config', planner_cfg,

        '--label', label,

    ]

    print('Running:', ' '.join(cmd))

    out = subprocess.run(cmd, text=True, capture_output=True, cwd=str(REPO_ROOT), check=False)

    if out.returncode != 0:

        print((out.stdout or '')[-2000:])

        print((out.stderr or '')[-2000:])

        raise RuntimeError(f'Leak probe failed for {label} with code {out.returncode}')

    print('[ok]', label)



run_leak(BASE_PLANNER, LEAK_BASE_LABEL)

run_leak(FT_PLANNER, LEAK_FT_LABEL)

run_leak(str(NULL_FT_PLANNER.relative_to(REPO_ROOT)), LEAK_NULL_LABEL)



def load_leak_ms(label: str):

    p = REPO_ROOT / 'runs' / label / 'multi_seed.json'

    obj = json.loads(p.read_text(encoding='utf-8'))

    return {r['policy']: r['metrics'] for r in obj.get('aggregate_policy_metrics', [])}



lb = load_leak_ms(LEAK_BASE_LABEL)

lf = load_leak_ms(LEAK_FT_LABEL)

ln = load_leak_ms(LEAK_NULL_LABEL)

pols = sorted(set(lb) & set(lf) & set(ln))

lfs = [lf[p]['task_success_rate']['mean'] - lb[p]['task_success_rate']['mean'] for p in pols]

lfi = [lf[p]['invalid_tool_call_rate']['mean'] - lb[p]['invalid_tool_call_rate']['mean'] for p in pols]

lns = [ln[p]['task_success_rate']['mean'] - lb[p]['task_success_rate']['mean'] for p in pols]

lni = [ln[p]['invalid_tool_call_rate']['mean'] - lb[p]['invalid_tool_call_rate']['mean'] for p in pols]

leak_ft_s = sum(lfs) / len(lfs) if lfs else 0.0

leak_ft_i = sum(lfi) / len(lfi) if lfi else 0.0

leak_null_s = sum(lns) / len(lns) if lns else 0.0

leak_null_i = sum(lni) / len(lni) if lni else 0.0

leak_adv_s = leak_ft_s - leak_null_s

leak_adv_i = leak_ft_i - leak_null_i

leak_invariant = abs(leak_adv_s) < 1e-6 and abs(leak_adv_i) < 1e-6



print('\n=== Leak-Probe Eval Summary ===')

print('toolset_id =', LEAK_TOOLSET_ID)

print('policies =', pols)

print(f'ft success delta: {leak_ft_s:+.4f}')

print(f'ft invalid delta: {leak_ft_i:+.4f}')

print(f'null success delta: {leak_null_s:+.4f}')

print(f'null invalid delta: {leak_null_i:+.4f}')

print(f'adapter advantage success: {leak_adv_s:+.4f}')

print(f'adapter advantage invalid: {leak_adv_i:+.4f}')

print('invariant across ft/null vs base:', leak_invariant)

Wrote leak-probe workload: /kaggle/working/tool-calling-reliability-benchmark/workloads/enriched/ambiguous_ops_leak_probe.json
Wrote leak-probe config: /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_leak_probe.json
Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_leak_probe.json --workload workloads/enriched/ambiguous_ops_leak_probe.json --seeds 11,22,33,44,55 --planner-config configs/planners/hf_qwen2_5_3b_base.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-leakprobe-base
[ok] toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-leakprobe-base
Running: uv run tcrb multi-seed --config /kaggle/working/tool-calling-reliability-benchmark/configs/toolset_reliability_leak_probe.json --workload workloads/enriched/ambiguous_ops_leak_probe.json --seeds 11,22,33,44,55 --planner-config configs/planners/hf_qwen2_5_3b_ft.json --label toolsetrel-hf-kaggle-qwen25-3b-sensitivity-v1-leakprobe-ft

In [16]:
from pathlib import Path

planner_dir = REPO_ROOT / 'configs' / 'planners'

available_planners = sorted(p.name for p in planner_dir.glob('*.json'))

print('Available planners:')

for name in available_planners:

    print('-', name)

Available planners:
- heuristic.json
- hf_qwen2_5_3b_base.json
- hf_qwen2_5_3b_ft.json
- policy_native.json
- stochastic_lowhalluc.json


In [12]:
publish_script = REPO_ROOT / 'scripts' / 'publish_kaggle_northstar_artifacts.py'

if not publish_script.exists():

    print('[publish] helper script not present in this runtime clone; skipping publish step.')

else:

    publish_cmd = [

        'uv', 'run', 'python', str(publish_script),

        '--dataset-slug', PUBLISH_DATASET_SLUG,

        '--title', PUBLISH_DATASET_TITLE,

        '--label-prefix', LABEL_PREFIX,

        '--repo-root', '.',

    ]

    print('Running:', ' '.join(publish_cmd))

    publish_res = subprocess.run(publish_cmd, text=True, capture_output=True, check=False)

    if publish_res.stdout:

        print(publish_res.stdout)

    if publish_res.returncode != 0:

        if publish_res.stderr:

            print(publish_res.stderr)

        raise RuntimeError(f'Publish failed with code {publish_res.returncode}')

[publish] helper script not present in this runtime clone; skipping publish step.


## Ready For Execution

Run cells top-to-bottom in Kaggle runtime to execute and publish the Toolset Reliability sensitivity study.